# Complaint Operations Intelligence — Data Cleaning & AI Preparation

## 1. Notebook Overview

This notebook prepares customer complaint data for downstream Artificial Intelligence (AI) processing.

The main objectives are to:

- Load the raw complaint dataset from the `complaint_flat.csv`.
- Inspect the structure and data quality of the source data.
- Remove fields that are not required for the downstream use case.
- Standardise date fields and derive useful time attributes.
- Explore important categorical fields such as product, issue, submission channel, and company response.
- Clean and normalise the free-text complaint descriptions.
- Create an AI-ready dataset containing complaints with usable textual content.
- Validate the resulting text and metadata quality.
- Save the final dataset as `CleanDataForAI.csv`.

> **Scope note:** The current notebook prepares and validates text for downstream AI/embedding use. It does **not** generate vector embeddings itself. Embedding generation should be treated as the next stage of the pipeline.


## 2. Load and Profile the Raw Dataset

The first step is to inspect the raw dataset before applying any transformations/cleaning.

The dataset is loaded into a pandas DataFrame using `low_memory=False` so that pandas does not infer mixed-type columns in separate chunks.

The initial profiling checks:

- Number of records and columns.
- Data types.
- Non-null counts.
- Missing-value percentages for every column.

This establishes the baseline data quality and helps determine which fields require cleaning or can be excluded.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("complaints_flat.csv", low_memory=False)

In [ ]:
df.info()

missing_percent = (df.isnull().mean()*100).round(2)

print(missing_percent.sort_values(ascending=False))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78313 entries, 0 to 78312
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   tags                       10900 non-null  object
 1   zip_code                   71556 non-null  object
 2   complaint_id               78313 non-null  int64 
 3   issue                      78313 non-null  object
 4   date_received              78313 non-null  object
 5   state                      76322 non-null  object
 6   consumer_disputed          42630 non-null  object
 7   product                    78313 non-null  object
 8   company_response           78313 non-null  object
 9   company                    78313 non-null  object
 10  submitted_via              78313 non-null  object
 11  date_sent_to_company       78313 non-null  object
 12  company_public_response    4 non-null      object
 13  sub_product                67742 non-null  object
 14  timely

## 3. Remove a Field with Extremely High Missingness

`company_public_response` is removed because the initial profiling shows that it contains almost no usable values (only 4 non-null records out of 78,313).

Keeping a field with this level of missingness would add little value to the downstream AI dataset and would increase unnecessary sparsity.

The remaining columns are retained for further assessment.


In [ ]:
clean_df = df.drop(columns=[
    "company_public_response"])

## 4. Standardise Date Fields

The two date fields are converted from their original string representation into pandas datetime values:

- `date_received`
- `date_sent_to_company`

Using a datetime representation makes the fields suitable for temporal analysis and prevents date values from being treated as ordinary text.


In [ ]:
clean_df["date_received"] = pd.to_datetime(clean_df["date_received"])

clean_df["date_sent_to_company"] = pd.to_datetime(clean_df["date_sent_to_company"])

## 5. Derive Time Features

The year and month are extracted from `date_received`.

These derived features provide a simple temporal representation that can later support trend analysis, filtering, aggregation, or AI-agent queries such as identifying complaint volumes by period.


In [ ]:
clean_df["year"] = clean_df["date_received"].dt.year
clean_df["month"] = clean_df["date_received"].dt.month

## 6. Inspect Product Distribution

The frequency distribution of `product` is examined to understand the types of financial products represented in the dataset.

This is an exploratory validation step rather than a transformation. It helps confirm that the product field contains meaningful categories and identifies dominant categories before the AI-ready dataset is created.

In [ ]:
clean_df["product"].value_counts()

product
Mortgage                                                                        22725
Checking or savings account                                                     12147
Credit card or prepaid card                                                     10797
Credit card                                                                     10373
Bank account or service                                                          9816
Credit reporting, credit repair services, or other personal consumer reports     4310
Debt collection                                                                  3201
Money transfer, virtual currency, or money service                               1500
Consumer Loan                                                                    1029
Vehicle loan or lease                                                             892
Student loan                                                                      866
Money transfers                               

## 7. Inspect Complaint Issue Distribution

The most frequent complaint issues are examined using `value_counts()`.

This provides an overview of the semantic categories already available in the source data and is useful for understanding the structure of the complaint taxonomy that can later support classification, retrieval, or agent reasoning.

In [ ]:
clean_df["issue"].value_counts().head(20)

issue
Loan modification,collection,foreclosure                                            9743
Managing an account                                                                 7410
Loan servicing, payments, escrow account                                            5360
Problem with a purchase shown on your statement                                     4055
Account opening, closing, or management                                             3879
Deposits and withdrawals                                                            3111
Incorrect information on your report                                                2714
Closing an account                                                                  1830
Billing disputes                                                                    1744
Trouble during payment process                                                      1673
Struggling to pay mortgage                                                          1552
Other          

## 8. Inspect Company Distribution

The company field is inspected to determine whether the dataset contains multiple companies or is focused on a single organisation.

The result shows that all records belong to `JPMORGAN CHASE & CO.`. Since the field therefore has no discriminating value within this dataset, it is removed later.

In [ ]:
clean_df["company"].value_counts().head(20)

company
JPMORGAN CHASE & CO.    78313
Name: count, dtype: int64

## 9. Inspect Submission Channels

The `submitted_via` field is analysed to understand how complaints entered the organisation.

The distribution provides useful operational context, such as the relative volume of complaints submitted through the web, referral, phone, postal mail, fax, and email channels.

In [ ]:
clean_df["submitted_via"].value_counts()

submitted_via
Web            47972
Referral       19550
Phone           6220
Postal mail     3486
Fax             1060
Email             25
Name: count, dtype: int64

## 10. Inspect Company Response Categories

The `company_response` field is profiled to understand the outcomes associated with complaints.

This field is retained because response categories may provide useful context for downstream analysis and AI-agent responses.

In [ ]:
clean_df["company_response"].value_counts()

company_response
Closed with explanation            60230
Closed with monetary relief        10129
Closed with non-monetary relief     4383
Closed without relief               2009
Closed with relief                   813
Closed                               628
In progress                          119
Untimely response                      2
Name: count, dtype: int64

## 11. Inspect Timeliness

The `timely` field is examined to understand whether complaints received a timely response.

This is an exploratory check. The field is later removed because the notebook's downstream AI dataset focuses on complaint content and operational metadata rather than this derived response-timeliness indicator.

In [ ]:
clean_df["timely"].value_counts()

timely
Yes    78217
No        96
Name: count, dtype: int64

## 12. Remove the Company Field

The `company` column is removed after the earlier analysis confirmed that every record refers to the same company.

Removing constant-value fields reduces unnecessary information in the downstream dataset without losing variation or predictive/contextual value.


In [ ]:
clean_df.drop(columns=["company"], inplace=True)

## 13. Remove the Timeliness Field

The `timely` field is removed from the AI-ready dataset.

The decision is based on the intended downstream use: the focus is on the complaint text, product/issue context, dates, submission information, and response metadata. The field is therefore not required for the prepared AI input.

In [ ]:
clean_df.drop(columns=["timely"], inplace=True)

## 14. Assess Availability of Complaint Text

`complaint_what_happened` is the key free-text field for downstream AI processing.

The notebook first measures how many records have no complaint narrative. This is important because an embedding model or text-based AI component requires meaningful textual input.

The initial dataset contains many records without a complaint narrative, so the notebook later creates a separate AI-ready subset containing only records with usable complaint text.

In [ ]:
print(clean_df["complaint_what_happened"].isna().sum())

57241


## 15. Sample Existing Complaint Narratives

A random sample of non-null complaint narratives is inspected manually.

This step helps understand the characteristics of the raw text before defining the cleaning rules. The sample demonstrates issues such as masked personal information, dates, monetary values, line breaks, inconsistent spacing, and other textual noise.

In [ ]:
clean_df["complaint_what_happened"].dropna().sample(5).tolist()

['I signed up for a Amazon credit card through Chase bank.. A month  or so later, I received a call from chase, wondering when I was going to pay my overdue bill.. I immediately paid the bill in full over the phone on the first call.. \nWhen I asked why I never received a bill, I was told that upon agreement, I agreed to it. That if I wanted a paper bill, I would have to call or go online to opt out of paperless billing. \nI received notification of a reported late payment on my credit report.. JCMPB card over 30 days late, with an overdue amount of $ XXXX..I dont believe Chase should hide the paperless agreement in the fine print at signup. Im sure it saves them money by not having to send bills out, but they should request the customer to go paperless. Instead, they force the customer to go paperless in the fine print that no one really reads, and then blames the customer for late payment when they receive no bill in the mail, to remind them to pay it.. \nIt seems predatory to me tha

## 16. Define the Complaint Text Cleaning Function

The `clean_complaint()` function applies deterministic text normalisation rules to the complaint narrative.

### Cleaning rules

1. **Missing values**
   - Missing text is converted to an empty string so the function can process every record safely.

2. **Lowercasing**
   - Text is converted to lowercase to reduce unnecessary variation caused by capitalisation.

3. **Money masking**
   - Monetary values represented using patterns such as `{$1600.00}` are replaced with the token `MONEY`.
   - This preserves the semantic fact that a monetary amount was mentioned without retaining the exact value.

4. **Date masking**
   - Masked or variable date patterns such as `XX/XX/XXXX` are replaced with the token `DATE`.
   - This reduces noise caused by specific dates while preserving the fact that a date was present.

5. **Removal of `XXXX` placeholders**
   - Repeated `X` placeholders are removed because they generally represent redacted information rather than meaningful language.

6. **URL removal**
   - URLs and web addresses are removed because they are generally not required for the semantic representation of the complaint.

7. **Whitespace normalisation**
   - Multiple spaces, tabs, and newline characters are collapsed into a single space.

8. **Final trimming**
   - Leading and trailing whitespace is removed.

The purpose is to preserve the semantic content of the complaint while reducing formatting noise and highly specific/redacted information before downstream AI processing.

In [ ]:
import re

def clean_complaint(text):
    if pd.isna(text):
        return ""

    # Lowercase
    text = text.lower()

    # Replace money amounts
    text = re.sub(r"\{\$.*?\}", " MONEY ", text)

    # Replace dates
    text = re.sub(r"\b(?:xx|x{2,})[/-](?:xx|x{2,})[/-](?:xxxx|\d{2,4})\b", " DATE ", text, flags=re.IGNORECASE)

    # Remove XXXX placeholders
    text = re.sub(r"\bX{2,}\b", " ", text, flags=re.IGNORECASE)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Replace multiple spaces/newlines
    text = re.sub(r"\s+", " ", text)

    return text.strip()

## 17. Generate the Clean Text Representation

The cleaning function is applied to every complaint narrative and stored in a new column named `clean_text`.

The original `complaint_what_happened` column is intentionally retained. This creates two representations:

- `complaint_what_happened`: original source text for traceability.
- `clean_text`: normalised text intended for downstream AI processing.

Keeping both versions makes the pipeline auditable because the cleaned representation can be compared with the original complaint.

In [41]:
clean_df["clean_text"] = clean_df["complaint_what_happened"].apply(clean_complaint)

## 18. Create the AI-Ready Complaint Subset

Only records with a non-null original complaint narrative are selected.

A second filter removes records where the resulting `clean_text` is empty after cleaning.

This produces `ai_df`, the working dataset for downstream AI processing. The filtering ensures that each retained record contains an actual textual complaint that can potentially be passed to an embedding model or other NLP component.

In [ ]:
ai_df = clean_df[clean_df["complaint_what_happened"].notna()].copy()
ai_df = ai_df[ai_df["clean_text"].str.strip() != ""].copy()
print(ai_df.shape)

(21072, 18)


## 19. Remove ZIP Code from the AI Dataset

`zip_code` is removed from the AI-ready dataset.

The downstream focus is complaint semantics and operational context rather than precise geographic postal information. Removing this field also reduces unnecessary location detail from the AI input dataset.

In [ ]:
ai_df.drop(columns=["zip_code"], inplace=True)

## 20. Inspect the AI-Ready Dataset

A small sample of the resulting `ai_df` is displayed to verify that:

- The original complaint is still available.
- `clean_text` has been created.
- Product and issue metadata remain associated with each complaint.
- Dates and other operational fields remain available.

This provides an early visual check that the cleaning process has preserved the relationship between complaint text and its metadata.

In [44]:
ai_df.head(5)

,tags,complaint_id,issue,date_received,state,consumer_disputed,product,company_response,submitted_via,date_sent_to_company,sub_product,complaint_what_happened,sub_issue,consumer_consent_provided,year,month,clean_text
1,Servicemember,3229299,Written notification about debt,2019-05-01 12:00:00-05:00,GA,NaN,Debt collection,Closed with explanation,Web,2019-05-01 12:00:00-05:00,Credit card debt,Good morning my name is XXXX XXXX and I apprec...,Didn't receive enough information to verify debt,Consent provided,2019,5,good morning my name is and i appreciate it if...
2,NaN,3199379,"Other features, terms, or problems",2019-04-02 12:00:00-05:00,TX,NaN,Credit card or prepaid card,Closed with explanation,Web,2019-04-02 12:00:00-05:00,General-purpose credit card or charge card,I upgraded my XXXX XXXX card in XX/XX/2018 and...,Problem with rewards from credit card,Consent provided,2019,4,i upgraded my card in DATE and was told by the...
10,NaN,3233499,Incorrect information on your report,2019-05-06 12:00:00-05:00,NY,NaN,"Credit reporting, credit repair services, or o...",Closed with explanation,Web,2019-05-06 12:00:00-05:00,Other personal consumer report,Chase Card was reported on XX/XX/2019. However...,Information belongs to someone else,Consent provided,2019,5,"chase card was reported on DATE . however, fra..."
11,NaN,3180294,Incorrect information on your report,2019-03-14 12:00:00-05:00,TX,NaN,"Credit reporting, credit repair services, or o...",Closed with explanation,Web,2019-03-15 12:00:00-05:00,Credit reporting,"On XX/XX/2018, while trying to book a XXXX XX...",Information belongs to someone else,Consent provided,2019,3,"on DATE , while trying to book a ticket, i cam..."
14,NaN,3224980,Managing an account,2019-04-27 12:00:00-05:00,CA,NaN,Checking or savings account,Closed with explanation,Web,2019-04-27 12:00:00-05:00,Checking account,my grand son give me check for {$1600.00} i de...,Funds not handled or disbursed as instructed,Consent provided,2019,4,my grand son give me check for MONEY i deposit...


## 21. Profile the AI-Ready Dataset

The structure of `ai_df` is inspected after the initial filtering and cleaning.

This confirms the resulting number of records, remaining columns, data types, and remaining missing values before further validation is performed.

In [45]:
ai_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21072 entries, 1 to 78312
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype                    
---  ------                     --------------  -----                    
 0   tags                       3816 non-null   object                   
 1   complaint_id               21072 non-null  int64                    
 2   issue                      21072 non-null  object                   
 3   date_received              21072 non-null  datetime64[ns, UTC-05:00]
 4   state                      20929 non-null  object                   
 5   consumer_disputed          6261 non-null   object                   
 6   product                    21072 non-null  object                   
 7   company_response           21072 non-null  object                   
 8   submitted_via              21072 non-null  object                   
 9   date_sent_to_company       21072 non-null  datetime64[ns, UTC-05:00]
 10  sub

## 22. Select the Final AI Dataset Columns

A controlled list of columns is defined to make the downstream dataset explicit and reproducible.

The selected fields include:

- Complaint identifier.
- Original and cleaned complaint text.
- Product and sub-product.
- Issue and sub-issue.
- Received and sent dates.
- State.
- Submission channel.
- Company response.
- Consumer dispute status.
- Tags.
- Consumer consent status.

Selecting columns explicitly prevents accidental retention of intermediate or irrelevant fields and establishes a clear schema for downstream AI components.

In [ ]:
ai_columns = [
    "complaint_id",
    "complaint_what_happened",
    "clean_text",
    "product",
    "sub_product",
    "issue",
    "sub_issue",
    "date_received",
    "date_sent_to_company",
    "state",
    "submitted_via",
    "company_response",
    "consumer_disputed",
    "tags",
    "consumer_consent_provided"
]

ai_df = ai_df[ai_columns].copy()

In [ ]:
ai_df.info()

In [48]:
ai_df.head()

,complaint_id,complaint_what_happened,clean_text,product,sub_product,issue,sub_issue,date_received,date_sent_to_company,state,submitted_via,company_response,consumer_disputed,tags,consumer_consent_provided
1,3229299,Good morning my name is XXXX XXXX and I apprec...,good morning my name is and i appreciate it if...,Debt collection,Credit card debt,Written notification about debt,Didn't receive enough information to verify debt,2019-05-01 12:00:00-05:00,2019-05-01 12:00:00-05:00,GA,Web,Closed with explanation,NaN,Servicemember,Consent provided
2,3199379,I upgraded my XXXX XXXX card in XX/XX/2018 and...,i upgraded my card in DATE and was told by the...,Credit card or prepaid card,General-purpose credit card or charge card,"Other features, terms, or problems",Problem with rewards from credit card,2019-04-02 12:00:00-05:00,2019-04-02 12:00:00-05:00,TX,Web,Closed with explanation,NaN,NaN,Consent provided
10,3233499,Chase Card was reported on XX/XX/2019. However...,"chase card was reported on DATE . however, fra...","Credit reporting, credit repair services, or o...",Other personal consumer report,Incorrect information on your report,Information belongs to someone else,2019-05-06 12:00:00-05:00,2019-05-06 12:00:00-05:00,NY,Web,Closed with explanation,NaN,NaN,Consent provided
11,3180294,"On XX/XX/2018, while trying to book a XXXX XX...","on DATE , while trying to book a ticket, i cam...","Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,2019-03-14 12:00:00-05:00,2019-03-15 12:00:00-05:00,TX,Web,Closed with explanation,NaN,NaN,Consent provided
14,3224980,my grand son give me check for {$1600.00} i de...,my grand son give me check for MONEY i deposit...,Checking or savings account,Checking account,Managing an account,Funds not handled or disbursed as instructed,2019-04-27 12:00:00-05:00,2019-04-27 12:00:00-05:00,CA,Web,Closed with explanation,NaN,NaN,Consent provided


## 23. Check Missing Values in the AI Dataset

Missing-value percentages are recalculated for the AI-ready dataset.

Unlike the original dataset, missingness in the remaining columns is evaluated in the context of the downstream AI use case. The key requirement is that `clean_text` must be populated for every retained complaint.

In [49]:
missing_percent = (ai_df.isnull().mean()*100).round(2)

print(missing_percent.sort_values(ascending=False))

tags                         81.89
consumer_disputed            70.29
sub_issue                    38.80
sub_product                  10.01
state                         0.68
complaint_id                  0.00
complaint_what_happened       0.00
clean_text                    0.00
product                       0.00
issue                         0.00
date_received                 0.00
date_sent_to_company          0.00
submitted_via                 0.00
company_response              0.00
consumer_consent_provided     0.00
dtype: float64


## 24. Validate Missing and Empty Clean Text

Two separate checks are performed:

- Missing `clean_text` values.
- Empty `clean_text` strings.

Both checks are important because a text column can be non-null while still containing an empty string. The expected result is zero missing and zero empty cleaned complaint texts in the AI-ready subset.


In [ ]:
print("Missing clean_text:",
      ai_df["clean_text"].isna().sum())

print("Empty clean_text:",
      (ai_df["clean_text"].str.strip() == "").sum())

Missing clean_text: 0
Empty clean_text: 0


## 25. Analyse Clean Text Length

Word-count statistics are calculated for `clean_text`.

The descriptive statistics help identify the typical and extreme lengths of complaint narratives. This is relevant to downstream NLP and embedding because unusually short or unusually long inputs may require additional consideration.


In [ ]:
print("Text length statistics:")
print(
    ai_df["clean_text"]
    .str.split()
    .str.len()
    .describe()
)

Text length statistics:
count    21072.000000
mean       246.302107
std        261.986831
min          1.000000
25%         94.000000
50%        176.000000
75%        311.000000
max       4967.000000
Name: clean_text, dtype: float64


## 26. Store Word Counts for Quality Analysis

A temporary `word_count` feature is created to support text-quality checks.

This field is used to investigate unusually short complaints and is an intermediate validation feature rather than a required final dataset field.


In [ ]:
ai_df["word_count"] = (
    ai_df["clean_text"]
    .str.split()
    .str.len()
)

## 27. Inspect the Shortest Complaints

The shortest complaints are inspected together with their original text, cleaned text, product, and issue.

The purpose is to determine whether very short texts are legitimate complaints or whether cleaning has accidentally removed too much information.


In [ ]:
ai_df.sort_values("word_count").head(30)[
    ["clean_text","complaint_what_happened", "word_count", "product", "issue"]
]

,clean_text,complaint_what_happened,word_count,product,issue
57428,#name?,#NAME?,1,"Credit reporting, credit repair services, or o...",Improper use of your report
142,chasemortgagehasautomaticallydeducted2mortgage...,ChaseMortgagehasautomaticallydeducted2mortgage...,1,Mortgage,Trouble during payment process
58343,unauthorized inquiry.,Unauthorized inquiry.,2,"Credit reporting, credit repair services, or o...",Improper use of your report
25726,unauthorized inquiry.,Unauthorized inquiry.,2,"Credit reporting, credit repair services, or o...",Improper use of your report
15056,unauthorized inquiry/inquiries.,Unauthorized inquiry/inquiries.,2,"Credit reporting, credit repair services, or o...",Improper use of your report
14901,never sent statements.,XXXX never sent statements.,3,Mortgage,Struggling to pay mortgage
4901,inaccurate information. outdated,Inaccurate information. Outdated,3,"Credit reporting, credit repair services, or o...",Incorrect information on your report
17820,inquiry not removed,inquiry not removed,3,"Credit reporting, credit repair services, or o...",Improper use of your report
3518,account mismanagement issue,account mismanagement issue,3,Credit card or prepaid card,Fees or interest
11632,unauthorized money missing,Unauthorized money missing,3,Checking or savings account,Problem caused by your funds being low


## 28. Check Duplicate Complaint IDs

Duplicate `complaint_id` values are counted.

The complaint ID is expected to identify an individual complaint. Checking duplicates helps ensure that the AI-ready dataset does not unintentionally contain repeated records with the same source identifier.


In [ ]:
print(
    "Duplicate complaint IDs:",
    ai_df["complaint_id"].duplicated().sum()
)

Duplicate complaint IDs: 0


## 29. Check Duplicate Original Complaint Texts

Duplicate original complaint narratives are counted.

Duplicate text does not necessarily mean duplicate complaints, but the check helps identify repeated narratives and provides additional information about dataset uniqueness.

In [ ]:
print(
    "Duplicate original texts:",
    ai_df["complaint_what_happened"].duplicated().sum()
)

Duplicate original texts: 142


## 30. Check Duplicate Cleaned Texts

Duplicate `clean_text` values are counted.

This is particularly relevant for semantic AI processing because different original records can potentially produce the same cleaned representation after masking and normalisation.

In [ ]:
print(
    "Duplicate clean texts:",
    ai_df["clean_text"].duplicated().sum()
)

Duplicate clean texts: 186


## 31. Inspect Very Short Complaints

Records with five words or fewer are displayed for manual review.

The goal is to distinguish genuinely short complaints from cases where the cleaning process may have removed too much information.


In [ ]:
ai_df[ai_df["word_count"] <= 5][
    ["clean_text", "word_count", "product", "issue"]
].sort_values("word_count")

,clean_text,word_count,product,issue
142,chasemortgagehasautomaticallydeducted2mortgage...,1,Mortgage,Trouble during payment process
57428,#name?,1,"Credit reporting, credit repair services, or o...",Improper use of your report
58343,unauthorized inquiry.,2,"Credit reporting, credit repair services, or o...",Improper use of your report
25726,unauthorized inquiry.,2,"Credit reporting, credit repair services, or o...",Improper use of your report
15056,unauthorized inquiry/inquiries.,2,"Credit reporting, credit repair services, or o...",Improper use of your report
17820,inquiry not removed,3,"Credit reporting, credit repair services, or o...",Improper use of your report
3518,account mismanagement issue,3,Credit card or prepaid card,Fees or interest
4901,inaccurate information. outdated,3,"Credit reporting, credit repair services, or o...",Incorrect information on your report
11632,unauthorized money missing,3,Checking or savings account,Problem caused by your funds being low
14901,never sent statements.,3,Mortgage,Struggling to pay mortgage


## 32. Quantify Very Short Complaints

The notebook reports:

- Total number of AI-ready complaints.
- Number of one-word complaints.
- Number of complaints with three words or fewer.
- Number of complaints with five words or fewer.

This quantifies the scale of the short-text issue identified during the previous inspection.


In [ ]:
ai_df["word_count"] = ai_df["clean_text"].str.split().str.len()

print("Total complaints:", len(ai_df))
print("1-word complaints:", (ai_df["word_count"] == 1).sum())
print("<= 3-word complaints:", (ai_df["word_count"] <= 3).sum())
print("<= 5-word complaints:", (ai_df["word_count"] <= 5).sum())

Total complaints: 21072
1-word complaints: 2
<= 3-word complaints: 10
<= 5-word complaints: 29


## 33. Review Short-Text Examples

The original and cleaned versions of very short complaints are displayed together with their product and issue.

This provides evidence for deciding whether these records should remain in the dataset or require additional cleaning rules.


In [ ]:
ai_df.loc[
    ai_df["word_count"] <= 5,
    ["complaint_id", "complaint_what_happened", "clean_text", "product", "issue"]
].to_string(index=False)

" complaint_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

## 34. Validate Categorical Coverage

The number of unique products, issues, and sub-issues is calculated.

This confirms the breadth of the complaint taxonomy retained in the AI-ready dataset and helps identify whether categorical metadata remains sufficiently diverse for downstream analysis.


In [ ]:
print("Unique products:", ai_df["product"].nunique())
print("Unique issues:", ai_df["issue"].nunique())
print("Unique sub-issues:", ai_df["sub_issue"].nunique())

Unique products: 17
Unique issues: 133
Unique sub-issues: 189


## 35. Review Product Frequencies

The most frequent product categories are displayed.

This provides a final sanity check on the product distribution after filtering to complaints with usable text.


In [ ]:
print("\nPRODUCTS")
print(ai_df["product"].value_counts().head(20))


PRODUCTS
product
Credit card or prepaid card                                                     5039
Checking or savings account                                                     4290
Mortgage                                                                        3246
Credit reporting, credit repair services, or other personal consumer reports    2087
Credit card                                                                     2070
Bank account or service                                                         1650
Debt collection                                                                  955
Money transfer, virtual currency, or money service                               797
Vehicle loan or lease                                                            403
Consumer Loan                                                                    240
Student loan                                                                     140
Money transfers                                

## 36. Review Issue Frequencies

The most frequent issue categories are displayed.

The result helps verify that the issue taxonomy remains intact and associated with the cleaned complaint narratives.


In [ ]:
print("\nISSUES")
print(ai_df["issue"].value_counts().head(20))


ISSUES
issue
Managing an account                                                                 2270
Problem with a purchase shown on your statement                                     1808
Incorrect information on your report                                                1202
Other features, terms, or problems                                                   753
Closing an account                                                                   747
Loan servicing, payments, escrow account                                             654
Problem with a lender or other company charging your account                         584
Loan modification,collection,foreclosure                                             582
Problem with a credit reporting company's investigation into an existing problem     546
Trouble during payment process                                                       526
Closing your account                                                                 524
Account

## 37. Review Sub-Issue Frequencies

The most frequent sub-issue categories are displayed.

This provides a deeper view of the complaint taxonomy and confirms that detailed issue metadata is available for downstream AI use where present.


In [ ]:
print("\nSUB-ISSUES")
print(ai_df["sub_issue"].value_counts().head(20))


SUB-ISSUES
sub_issue
Credit card company isn't resolving a dispute about a purchase on your statement    1260
Company closed your account                                                          890
Deposits and withdrawals                                                             668
Problem using a debit or ATM card                                                    511
Card was charged for something you did not purchase with the card                    498
Information belongs to someone else                                                  485
Transaction was not authorized                                                       463
Credit inquiries on your report that you don't recognize                             379
Their investigation did not fix an error on your report                              331
Account status incorrect                                                             310
Other problem                                                                        305

## 38. Manual Sample Validation

A reproducible random sample of 20 records is selected using `random_state=42`.

The sample compares:

- Original complaint text.
- Cleaned complaint text.
- Product.
- Issue.
- Sub-issue.

Using a fixed random state makes the sample reproducible, which is useful when reviewing or demonstrating the cleaning process.


In [ ]:
ai_df[
    ["complaint_what_happened", "clean_text", "product", "issue","sub_issue"]
].sample(20, random_state=42)

,complaint_what_happened,clean_text,product,issue,sub_issue
30957,OXXXX XXXX paid JP Morgan Chase {$1200.00} for...,oxxxx paid jp morgan chase MONEY for mortgage ...,Mortgage,Trouble during payment process,NaN
58320,I lost my credit card ( a Chase Sapphire Reser...,i lost my credit card ( a chase sapphire reser...,Credit card or prepaid card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...
61282,"In 2005, I went to XXXX XXXX XXXX. They direct...","in 2005, i went to . they directed me to bank ...",Student loan,Can't repay my loan,Can't get flexible payment options
40500,I applied for and received a Chase Sapphire Pr...,i applied for and received a chase sapphire pr...,Credit card or prepaid card,"Advertising and marketing, including promotion...",Didn't receive advertised or promotional terms
54027,With Chase a {$620.00} charge ( XXXX 2015 ) wa...,with chase a MONEY charge ( 2015 ) was origina...,Credit card,Transaction issue,NaN
64903,I have advised Chase on a few occasions that I...,i have advised chase on a few occasions that i...,Debt collection,Communication tactics,Called after sent written cease of comm
12319,My issues is with JP Morgan Chase. On XX/XX/XX...,my issues is with jp morgan chase. on DATE the...,Checking or savings account,Closing an account,Company closed your account
21744,On XX/XX/XXXX there was a cash withdrawal of {...,on DATE there was a cash withdrawal of MONEY m...,Checking or savings account,Managing an account,Banking errors
28545,On or about XX/XX/XXXX I experienced fraudulen...,on or about DATE i experienced fraudulent acti...,"Credit reporting, credit repair services, or o...",Incorrect information on your report,Account status incorrect
10449,I deposited a check that is actually with chas...,i deposited a check that is actually with chas...,Checking or savings account,Managing an account,Deposits and withdrawals


## 39. Detect Spreadsheet/Error-Style `#NAME?` Values

The notebook checks for records where the entire original complaint field is `#NAME?`.

This is a targeted data-quality check for values that look like spreadsheet formula errors rather than genuine complaint narratives.


In [ ]:
ai_df[
    ai_df["complaint_what_happened"].astype(str).str.strip().str.upper() == "#NAME?"
].shape

(1, 16)

## 40. Investigate Suspicious Text Compression

The notebook compares original and cleaned text lengths and searches for cases where the original complaint is reasonably long but the cleaned result contains three words or fewer.

This is an important validation step because it can reveal cases where a cleaning rule has unintentionally removed substantial amounts of meaningful text.

In [ ]:
ai_df["original_length"] = (
    ai_df["complaint_what_happened"]
    .astype(str)
    .str.len()
)

ai_df["clean_length"] = ai_df["clean_text"].str.len()

ai_df[
    (ai_df["original_length"] > 50) &
    (ai_df["word_count"] <= 3)
][
    ["complaint_what_happened", "clean_text", "word_count"]
].head(20)

,complaint_what_happened,clean_text,word_count
142,ChaseMortgagehasautomaticallydeducted2mortgage...,chasemortgagehasautomaticallydeducted2mortgage...,1


## 41. Remove Invalid `#NAME?` Records

Records whose complete complaint text is `#NAME?` are removed.

These values do not represent meaningful complaint narratives and therefore should not be passed to downstream text-processing or embedding components.


In [ ]:
ai_df = ai_df[
    ai_df["complaint_what_happened"].astype(str).str.strip().str.upper() != "#NAME?"
].copy()

In [ ]:
ai_df[
    ai_df["complaint_what_happened"].astype(str).str.strip().str.upper() == "#NAME?"
].shape

(0, 18)

## 42. Check for Texts Without Whitespace

The notebook creates a temporary `has_spaces` flag indicating whether the original complaint contains whitespace.

This is used to identify potentially malformed text values such as a long sequence without normal word separation.


In [ ]:
ai_df["has_spaces"] = (
    ai_df["complaint_what_happened"]
    .astype(str)
    .str.contains(r"\s")
)

ai_df["has_spaces"].value_counts()

has_spaces
True     21068
False        3
Name: count, dtype: int64

In [ ]:
ai_df[
    ~ai_df["has_spaces"]
][
    ["complaint_what_happened", "clean_text"]
].head(20)

,complaint_what_happened,clean_text
142,ChaseMortgagehasautomaticallydeducted2mortgage...,chasemortgagehasautomaticallydeducted2mortgage...
8588,IacceptedapromotionalofferonmyChaseSapphirecre...,iacceptedapromotionalofferonmychasesapphirecre...
15185,Throughoutthelastfewmonthswehavehadawebsite(XX...,throughoutthelastfewmonthswehavehadawebsite( )...


## 43. Final Dataset Profiling Before Cleanup

The complete structure of `ai_df` is inspected after the quality checks.

At this point, the dataset contains both the final fields and several temporary validation columns used during the cleaning process.


In [71]:
ai_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21071 entries, 1 to 78312
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype                    
---  ------                     --------------  -----                    
 0   complaint_id               21071 non-null  int64                    
 1   complaint_what_happened    21071 non-null  object                   
 2   clean_text                 21071 non-null  object                   
 3   product                    21071 non-null  object                   
 4   sub_product                18962 non-null  object                   
 5   issue                      21071 non-null  object                   
 6   sub_issue                  12895 non-null  object                   
 7   date_received              21071 non-null  datetime64[ns, UTC-05:00]
 8   date_sent_to_company       21071 non-null  datetime64[ns, UTC-05:00]
 9   state                      20928 non-null  object                   
 10  sub

## 44. Remove Temporary Validation Columns

The temporary validation columns are removed:

- `word_count`
- `original_length`
- `clean_length`
- `has_spaces`

These fields supported quality-control analysis but are not part of the intended downstream AI dataset schema.


In [72]:
ai_df=ai_df.drop(columns=["word_count","original_length","clean_length","has_spaces"])
ai_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21071 entries, 1 to 78312
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype                    
---  ------                     --------------  -----                    
 0   complaint_id               21071 non-null  int64                    
 1   complaint_what_happened    21071 non-null  object                   
 2   clean_text                 21071 non-null  object                   
 3   product                    21071 non-null  object                   
 4   sub_product                18962 non-null  object                   
 5   issue                      21071 non-null  object                   
 6   sub_issue                  12895 non-null  object                   
 7   date_received              21071 non-null  datetime64[ns, UTC-05:00]
 8   date_sent_to_company       21071 non-null  datetime64[ns, UTC-05:00]
 9   state                      20928 non-null  object                   
 10  sub

In [ ]:
ai_df[
    ai_df["complaint_what_happened"].astype(str).str.strip().str.upper() == "#NAME?"
].shape

(0, 15)

## 45. Final Manual Review Sample

A final reproducible sample of 30 records is created containing the original complaint, cleaned complaint, product, issue, and sub-issue.

This is the final qualitative review of the prepared dataset before it is persisted to disk.


In [ ]:
sample_30 = ai_df[
    [
        "complaint_what_happened",
        "clean_text",
        "product",
        "issue",
        "sub_issue"
    ]
].sample(30, random_state=42)

sample_30

,complaint_what_happened,clean_text,product,issue,sub_issue
30957,OXXXX XXXX paid JP Morgan Chase {$1200.00} for...,oxxxx paid jp morgan chase MONEY for mortgage ...,Mortgage,Trouble during payment process,NaN
58321,I just explained the situation on previous pag...,i just explained the situation on previous pag...,Checking or savings account,Managing an account,Problem using a debit or ATM card
61283,I did a balance transfer using promotion check...,i did a balance transfer using promotion check...,Credit card,Payoff process,NaN
40500,I applied for and received a Chase Sapphire Pr...,i applied for and received a chase sapphire pr...,Credit card or prepaid card,"Advertising and marketing, including promotion...",Didn't receive advertised or promotional terms
54027,With Chase a {$620.00} charge ( XXXX 2015 ) wa...,with chase a MONEY charge ( 2015 ) was origina...,Credit card,Transaction issue,NaN
64904,I am writing about the privacy notice I receiv...,i am writing about the privacy notice i receiv...,Credit card,Privacy,NaN
12319,My issues is with JP Morgan Chase. On XX/XX/XX...,my issues is with jp morgan chase. on DATE the...,Checking or savings account,Closing an account,Company closed your account
21744,On XX/XX/XXXX there was a cash withdrawal of {...,on DATE there was a cash withdrawal of MONEY m...,Checking or savings account,Managing an account,Banking errors
28545,On or about XX/XX/XXXX I experienced fraudulen...,on or about DATE i experienced fraudulent acti...,"Credit reporting, credit repair services, or o...",Incorrect information on your report,Account status incorrect
10449,I deposited a check that is actually with chas...,i deposited a check that is actually with chas...,Checking or savings account,Managing an account,Deposits and withdrawals


## 46. Export the AI-Ready Dataset

The final cleaned dataset is saved as:

`CleanDataForAI.csv`

The output contains the prepared complaint narratives and their associated metadata and is intended to serve as the input to the next stage of the AI pipeline.


In [76]:
ai_df.to_csv("CleanDataForAI.csv",index=False)